In [0]:
from pyspark.sql.functions import col, current_date,date_format, trim, lower,max
import datetime
from datetime import date
import pandas as pd
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

####1.Calling Params notebook 

In [0]:
%run ../utils/params

#### 2.Calling logging notebook

In [0]:
%run ../utils/loggingNotebook

#### 3.Calling utilsMethod

In [0]:
%run ../utils/utilsMethods

In [0]:
notebookPath = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebookName = notebookPath.split("/")[-1]
jobID = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("jobId")
properties = { "custom_dimensions": {"notebookName": f"{notebookName}", "jobID": f"{jobID}"}}

#### 4.Reading ruleObjectMapping file and dataQualityResult table in Dataframes.

In [0]:
dataQualityResultDf = spark.sql(f"select * from {dataQualityTable}").withColumn("dqStatus", lower("dqStatus"))\
                       .withColumn("alertStatus", lower("alertStatus")).filter(col("alertStatus") != "yes")

ruleObjectMappingDf = (spark.read.format("csv").option("header", True).option("inferSchema", True).load(ruleObjectMappingPath)\
.withColumn("criticality", lower("criticality")))

#### 5.Daily DQ report based on failure and criticality

In [0]:
maxdqEndTs = dataQualityResultDf.select(max("dqEndTs")).collect()[0][0].strftime('%Y-%m-%d')
joinedDf = dataQualityResultDf.join(ruleObjectMappingDf.select(col("mappingID"), col("criticality")),"mappingID","inner")\
.filter((col("dqStatus") == "failure") & (date_format("dqEndTs", "yyyy-MM-dd") == maxdqEndTs))

In [0]:
try:
    if joinedDf.count() == 0:
        logger.info("No data in dataQualityResult table for the current date. Exiting notebook.",extra=properties)

    else:

        if joinedDf.filter(col("criticality") == "medium").count() > 0:
        
            borderStyle = "1px solid black"
            newDf = joinedDf.filter(col("criticality") == "medium").drop(col('dqErrorID'),col('alertStatus'),col('mappingID'),col('criticality'))


            # generate HTML table with styled rows
            htmlTable = (newDf.select(newDf.columns).toPandas().to_html(classes=["table", "table-bordered"], escape=False, index=False))

            # add custom CSS style for table header
            cssStyle = """
            <style>
                    .table-header {
                        background-color: purple;
                        color: white;
                        border: 1px solid black;
                    }
                    th {
                        text-align: center;
                    }
            </style>
            """

            htmlTable = htmlTable.replace("<thead>", '<thead class="table-header">')
            htmlTable = cssStyle + htmlTable

            sendEmailAlert(senderEmail, recipientEmail, emailSubject, html=htmlTable)


            logger.info("Email Aerts has been sent for Criticality level 'medium' and dqStatus 'failure' for dataQualityResult table ",extra=properties)
                

            for row in (joinedDf.filter(col("criticality") == "medium").select(col("mappingID")).distinct().collect()):
                
                mappingID = row.mappingID

                update_query = f"""UPDATE {dataQualityTable} SET alertStatus = "yes" WHERE mappingID = '{mappingID}'"""
                spark.sql(update_query)
            logger.info("Email Aerts has been triggered successfully and updated the flag of alertStatus to Yes ",extra=properties)


        if joinedDf.filter(col("criticality") == "low").count() > 0:
        
            logger.warning("Critical issue detected. Please investigate the dataQualityResult table.",extra=properties)
       
        
except exception as e:
    logger.exception(f"An Error occured while sending email:{str(e)}",extra=properties)
    raise e
    